# KG NER integration

This notebook explains how Named Entity Recognition (NER) is integrated with the Stark-Prime Knowledge Graph (KG).

Typically, descriptions of perturbations and cellular contexts do not explicitly provide node names that match the Stark-Prime KG.

To address this, we implemented an NER-based approach using HunFlair2 (https://flairnlp.github.io/flair/master/tutorial/tutorial-hunflair2/overview.html) and developed enhanced matching techniques.

The workflow proceeds as follows:

- Entity Extraction: The NER tool identifies biological entities (e.g., proteins, genes, diseases).

- Exact Matching: If an extracted entity exactly matches a node name in the KG, we directly query the corresponding node.

- Synonym Matching: If no exact match exists, we perform an external database search to retrieve synonyms or alternative names for the entity, which are then used to identify the corresponding KG node.

This integration ensures that free-text biological descriptions can be systematically linked to structured KG representations, enabling more accurate and comprehensive knowledge retrieval.

## Load data

In [1]:
from explain.kg.kg import KnowledgeGraph
from explain.util import extract_entity_from_text

import os
import sys

# Set PYTHONPATH to HOOKE-EXPLAIN root if not already set
os.environ['PYTHONPATH'] = os.path.join('../../../')

/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

kg = KnowledgeGraph()
# text: you can use the your own text (In my case, perturbation)
text = """
"perturbation": {
            "context": {
                "perturbation_type": "loss-of-function",
                "description": "CRISPR knockdown of the TSC2 gene",
                "cell_type": "N/A",
                "disease_model": "Tuberous sclerosis"
            },
            "perturbation": {
                "type": "chemical",
                "smiles": "CS(=O)(=O)N1CCN(CC1)CC2=CC3=C(S2)C(=NC(=N3)C4=C5C=NNC5=CC=C4)N6CCOCC6",
                "name": "GDC-0941",
                "target": "MTOR",
                "moa_type": "inhibitor"
            }
        }
"""

Loading embeddings from /rxrx/data/user/hamed.shirzad/outgoing/stark_prime_kg/pritamdeka/S-PubMedBERT-MS-MARCO/node_embeddings.pt


/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


## Entity extraction

In [3]:
# entities extracted from text ({'entity': 'tag'})
entities = extract_entity_from_text(text)
entities

2025-08-18 13:30:36,245 SequenceTagger predicts: Dictionary with 21 tags: O, S-Chemical, B-Chemical, E-Chemical, I-Chemical, S-Gene, B-Gene, E-Gene, I-Gene, S-Disease, B-Disease, E-Disease, I-Disease, S-Species, B-Species, E-Species, I-Species, S-CellLine, B-CellLine, E-CellLine, I-CellLine


/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'TSC2': 'Gene',
 'Tuberous sclerosis': 'Disease',
 'GDC-0941': 'Chemical',
 'MTOR': 'Gene'}

## Exact matching 

In [4]:
kg_node_info = kg.node_info
kg_node_name_dict = {node['name']: idx for idx, node in kg_node_info.items()}
# result_node_list: corresponding node in StarkPrimeKG
exact_matching_result = []

In [5]:
for entity in entities.keys():
    if entity in kg_node_name_dict.keys():
        exact_matching_result.append({'entity': entity, 'node': kg_node_name_dict[entity]})

In [6]:
exact_matching_result

[{'entity': 'TSC2', 'node': '8372'}, {'entity': 'MTOR', 'node': '1558'}]

## Synonym matching
Note: Here, exact matching is integrated with synonym matching (first search for exact matching nodes and then find the synonyms for unmatched entities)

In [7]:
from explain.kg.kg_utils import get_ner_node_index

synonym_matching_result, unmatching_entities = get_ner_node_index(entities, kg)

/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/src/explain/kg/kg_utils.py:217: UserWarning: Could not find the node index for GDC-0941 in the KG.
  warnings.warn(f"Could not find the node index for {entity} in the KG.")


In [8]:
synonym_matching_result

[{'entity': 'TSC2', 'node': 8372},
 {'entity': 'Tuberous sclerosis', 'node': 30967},
 {'entity': 'MTOR', 'node': 1558}]

In [9]:
unmatching_entities

['GDC-0941']

In [19]:
# for unmatching entities, I am simply using the find_similar_nodes function to find the closest nodes in the KG

node_list = []
for unmatching_entity in unmatching_entities:
    node_list.extend(list(kg.find_similar_nodes(unmatching_entity, k=1).keys()))

/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [21]:
node_list

[16611]